# [실습] LangChain의 다양한 채팅 템플릿



실제 어플리케이션 환경에서는 프롬프트를 매번 다시 쓰는 대신,  
프롬프트의 템플릿을 구성하고, 입력 변수의 공간을 할당하여 일관성 있는 입력을 전달합니다.  


LangChain은 다양한 형태의 프롬프트 템플릿 기능을 지원합니다.

## 실습 환경 설정

기본 라이브러리를 설치합니다.

In [1]:
!pip install rich pandas langchain_community openai langchain langchain_openai langchain_google_genai -q

환경 변수 파일을 불러옵니다.

In [2]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

LLM을 정의합니다.   
API 키가 필요하므로, 실습 환경에 맞게 선택해 주세요.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
llm = ChatOpenAI(model='gpt-5-mini', temperature=0.3, max_tokens=8192)
llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, max_tokens=8192, 
                                rate_limiter=rate_limiter)

response = llm_gemini.invoke("안녕?")
response

AIMessage(content='안녕하세요! 😊', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--4917630a-73b0-477a-aef6-a40595deaa94-0', usage_metadata={'input_tokens': 4, 'output_tokens': 49, 'total_tokens': 53, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 46}})

In [4]:
from rich import print as rprint
rprint(response)

AIMessage(
    content='안녕하세요! 😊',
    additional_kwargs={},
    response_metadata={
        'prompt_feedback': {'block_reason': 0, 'safety_ratings': []},
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': []
    },
    id='run--4917630a-73b0-477a-aef6-a40595deaa94-0',
    usage_metadata={
        'input_tokens': 4,
        'output_tokens': 49,
        'total_tokens': 53,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 46}
    }
)

## LangChain의 Prompt

랭체인에서 LLM에 프롬프트를 입력하는 방법은 아래 3가지가 있습니다.

1. 문자열 그대로 입력하기 (이전 방법과 동일하게)
2. 메시지 클래스 리스트 입력하기
3. 프롬프트 템플릿 구성하기

### 1) 문자열 그대로 입력하기   

문자열을 `invoke()`를 통해 전달합니다.   

In [5]:
prompt = '''
역사상 가장 매력적이었던 악역이 나오는 영화 5개를 알려주세요.
주연 배우와 명대사, 매력적인 이유도 한 줄로 설명하세요.
출력의 마지막에는 전체 내용을 표로 표현하세요.'''

result = llm.stream(prompt)
for response in result:
    print(response.content, end='')


아래는 역사상 가장 매력적이라고 꼽히는 악역이 등장하는 영화 5편과 각 항목별 정보입니다.

1) 양들의 침묵 (The Silence of the Lambs, 1991)  
- 주연 배우(악역 연기): 앤서니 홉킨스 (Anthony Hopkins)  
- 명대사: "안녕, 클라리스."  
- 매력 이유(한 줄): 냉철한 지성과 세련된 매너로 공포를 우아하게 만드는 지적 카리스마.

2) 다크 나이트 (The Dark Knight, 2008)  
- 주연 배우(악역 연기): 히스 레저 (Heath Ledger)  
- 명대사: "왜 그렇게 진지해?"  
- 매력 이유(한 줄): 예측 불가능한 혼돈 철학과 광기 어린 연기로 관객을 사로잡음.

3) 스타워즈: 제국의 역습 (The Empire Strikes Back, 1980)  
- 주연 배우(악역 연기): 데이비드 프라우스(신체)/제임스 얼 존스(목소리)  
- 명대사: "아니, 내가 네 아버지다."  
- 매력 이유(한 줄): 거대한 권위와 비극적 서사가 결합된 상징적 악역의 존재감.

4) 노인을 위한 나라는 없다 (No Country for Old Men, 2007)  
- 주연 배우(악역 연기): 하비에르 바르뎀 (Javier Bardem)  
- 명대사: "Call it." (결정을 내려라)  
- 매력 이유(한 줄): 운명처럼 냉혹하고 무자비한 태도로 공포를 서늘하게 만드는 존재감.

5) 바스터즈: 거친 녀석들 (Inglourious Basterds, 2009)  
- 주연 배우(악역 연기): 크리스토프 발츠 (Christoph Waltz)  
- 명대사: "Au revoir."  
- 매력 이유(한 줄): 매너와 교활함을 겸비한 대사 처리와 눈빛으로 관객을 압도.

아래는 위 내용을 요약한 표입니다.

| 영화 (원제, 연도) | 악역(역할) | 주연 배우(악역 연기) | 명대사 | 매력 포인트 (한 줄) |
|---|---:|---|---|---|
| 양들의 침묵 (The Silence of t

In [7]:
result = llm_gemini.stream(prompt)
for response in result:
    print(response.content, end='')

역사상 가장 매력적이었던 악역들은 단순히 악행을 저지르는 것을 넘어, 탁월한 지능, 압도적인 카리스마, 예측 불가능한 광기, 혹은 복합적인 내면으로 관객들을 사로잡았습니다. 다음은 그러한 매력적인 악역이 등장하는 영화 5편입니다.

---

### 역사상 가장 매력적이었던 악역이 나오는 영화 5편

1.  **영화 제목:** 다크 나이트 (The Dark Knight, 2008)
    *   **악역 이름:** 조커 (Joker)
    *   **주연 배우:** 히스 레저 (Heath Ledger)
    *   **명대사:** "Why so serious?" (왜 그렇게 심각해?)
    *   **매력적인 이유:** 예측 불가능한 광기와 혼돈을 즐기며, 탁월한 지능과 카리스마로 고담시를 조롱하고 사람들을 매혹시킨다.

2.  **영화 제목:** 양들의 침묵 (The Silence of the Lambs, 1991)
    *   **악역 이름:** 한니발 렉터 (Dr. Hannibal Lecter)
    *   **주연 배우:** 안소니 홉킨스 (Anthony Hopkins)
    *   **명대사:** "A census taker once tried to test me. I ate his liver with some fava beans and a nice Chianti." (어떤 인구조사원이 날 시험하려 했지. 나는 그의 간을 파바 콩과 맛있는 키안티 와인과 함께 먹었어.)
    *   **매력적인 이유:** 고상하고 지적인 태도 뒤에 숨겨진 잔혹함, 그리고 상대를 꿰뚫어 보는 통찰력으로 압도적인 존재감을 발휘한다.

3.  **영화 제목:** 바스터즈: 거친 녀석들 (Inglourious Basterds, 2009)
    *   **악역 이름:** 한스 란다 (Col. Hans Landa)
    *   **주연 배우:** 크리스토프 발츠 (Christoph Waltz)
    *   **명대사:** "Ooh, that's a bingo!" 

### 2) Message 클래스 전달하기   
클래스를 직접 생성하고 전달합니다.  
`HumanMessage, SystemMessage, AIMessage` 클래스의 리스트를 전달하면 됩니다.

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages=[]

messages.append(SystemMessage('당신은 매우 창의력이 뛰어납니다.'))
messages.append(HumanMessage('엔비디아와 XAI가 합병한다면 어떤 이름이 좋을까요?'))

result = llm.invoke(messages)
rprint(result)


AIMessage(
    content="좋은 질문이에요 — 합병 회사 이름은 목표(기업용/컨슈머/연구), 톤(권위적/친근/미래지향)과 
IP(상표·도메인) 제약에 따라 달라집니다. 아래는 용도별로 나눈 후보들과 짧은 설명입니다. 원하시는 톤을 알려주시면 
후보를 더 좁혀 드릴게요.\n\n브랜더블·짧고 기억하기 쉬운 이름\n- NVIDAI (엔비다이): NVIDIA + AI를 직관적으로 
결합.\n- NVX (엔브이엑스): 간결하고 기술 브랜드 느낌.\n- XNVD (엑스엔브이디): 두 이름의 이니셜 결합형, 모던함.\n- 
Nuxel (녹셀): 'new'/'nexus'와 'pixel' 느낌의 조어.\n- Xyra (싸이라): 발음이 쉽고 글로벌 친화적.\n- Vidu (비두): 
짧고 유니크한 상표형 조어.\n\n설명형·기업·신뢰 중심\n- FusionAI (퓨전에이아이): 두 회사의 결합과 AI 중점성 강조.\n-
NeuralFusion (뉴럴퓨전): 뉴럴 테크와 통합 플랫폼 이미지.\n- ComputeX AI (컴퓨트엑스에이아이): 고성능 컴퓨팅 + 
X(설명/확장성) 강조.\n- CoreVision AI (코어비전에이아이): 비전·컴퓨팅 핵심을 제공하는 기업 이미지.\n- SynapseX 
(시냅스엑스): 신경망·연결성·확장성 암시.\n- UnifiedAI (유니파이드에이아이): 통합 플랫폼·생태계 이미지를 
전달.\n\n미래지향·감성적\n- Meridian AI (메리디안에이아이): 미래 지향적·글로벌 포지셔닝.\n- HelixAI 
(헬릭스에이아이): 발전·진화 이미지.\n- PrismX (프리즘엑스): 다각적 가능성과 시각화 중심.\n- NebulaAI 
(네뷸라에이아이): 광범위한 혁신·연구 느낌.\n- AetherX (에테르엑스): 첨단·미래 과학 이미지.\n- LatticeAI 
(라티스에이아이): 구조적·확장 가능한 플랫폼 암시.\n\n플랫폼/제품 라인에 어울리는 이름\n- Omnistream (옴니스트림): 
데이터 흐름·스트리밍 플랫폼에 적합.\n- MatrixForge (매트릭스포지): 모델 훈련·시뮬레이션 플랫폼 이미지.\n- 
InsightGrid (인사이트그리드): 분석·인사이트 파이프라인 강조.\n- GraphCore (그래프코어): 그래프·네트워크 계산 
중심.\n- EdgeWeave (엣지위브): 엣지 AI·분산 컴퓨팅에 적합.\n- VisionStack (비전스택): 비전 관련 소프트웨어/하드웨어
스택 명칭에 적합.\n\n다음 권장사항\n- 후보 몇 개를 골라 상표·도메인(예: .com)·SNS 핸들 사용 가능 여부를 
확인하세요.\n- 글로벌 발음·현지 의미(다른 언어에서의 부정적 의미) 테스트를 권합니다.\n원하시면 선호 톤(예: 
기업적·친근·혁신적)과 타깃(기업 고객 vs 일반 사용자)을 알려 주세요. 그에 맞춰 5개로 압축하고 도메인·상표 가능성도 
간단히 확인해 드릴게요.",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 1977,
            'prompt_tokens': 41,
            'total_tokens': 2018,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 1152,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-C7g9kwLGRQYUWQlNWhk9hdx7GLujO',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--239035a1-54f8-4348-b9e6-1738496c9cda-0',
    usage_metadata={
        'input_tokens': 41,
        'output_tokens': 1977,
        'total_tokens': 2018,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 1152}
    }
)

AIMessage를 함께 전달하는 방식으로, 멀티-턴 대화를 수행할 수 있습니다.

In [9]:
messages.append(result) # AIMessage 추가
messages.append(HumanMessage('이 중에서 당신이 꼽은 가장 좋은 이름은 뭔가요?'))

result = llm.invoke(messages)
rprint(result)

AIMessage(
    content='제일 좋은 후보 하나만 고르라면 저는 "NeuralFusion"을 추천합니다.\n\n이유 요약:\n- 의미가 직관적: 두 
회사의 결합(퓨전)과 뉴럴/딥러닝 핵심 역량을 동시에 전달합니다.\n- 브랜드·제품 범용성: 엔터프라이즈 솔루션, 
하드웨어(가속기), 소프트웨어 플랫폼 등 모두에 어울립니다.\n- 발음·이미지: 글로벌 시장에서 전문적이고 미래지향적인 
인상을 줍니다.\n\n주의사항(간단):\n- 상표·도메인(.com) 가능성은 별도 확인이 필요합니다.\n- 원하면 짧은 
변형(NeuraFuse, NeuroFusion)이나 한국어 표기안, 도메인·상표 조회까지 도와드릴게요. 어느 쪽으로 진행할까요?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 518,
            'prompt_tokens': 879,
            'total_tokens': 1397,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 320,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-C7gAGOtsobP0NIqvGmDnBmXKq9x82',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--c2874061-6d69-49d3-9380-8a6327f12287-0',
    usage_metadata={
        'input_tokens': 879,
        'output_tokens': 518,
        'total_tokens': 1397,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 320}
    }
)

### 3. Prompt Template



In [10]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate, FewShotPromptTemplate, FewShotChatMessagePromptTemplate

PromptTemplate을 이용하여, 프롬프트의 기본적인 형태를 만들 수 있습니다.   


In [11]:
prompt = PromptTemplate(
    template='''
당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: {article}

'''
)

print(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))


당신은 OpenAI에 경쟁적인 Google을 대표하며,
논문 설명의 전문가입니다. 주어진 논문이나 기술에 대해서 매우 유머러스하고 통찰력이 있지만 때론 비판적인
5문장 길이의 브리핑을 작성해 주세요.
이모지를 많이 추가하세요.
---

논문: 트랜스포머 디코더 기반의 모델인 GPT-1




In [12]:
result = llm_gemini.invoke(prompt.format(article='트랜스포머 디코더 기반의 모델인 GPT-1'))

rprint(result)

AIMessage(
    content="아, GPT-1이라! 🤖 OpenAI가 드디어 *우리의* 혁신적인 트랜스포머 아키텍처, 특히 디코더 부분을 가지고 
놀기 시작했다는 걸 보여준 논문이죠. 😉 대규모 텍스트 코퍼스에 사전 학습시키는 것이 얼마나 강력한지, *음, 우리도 
이미 알고 있었지만* 그들이 직접 증명해줬네요! 📚✨\n\n이 'Generative Pre-trained Transformer'는 사실상 현대 LLM의 
귀여운 👶, 조금은 어설픈 첫걸음이었달까요? 당시엔 획기적이었지만, 지금 보면 그 '지능'은 열정 넘치지만 길을 잘 잃는 
인턴 수준이랄까. 😅\n\n그래도 *우리* 트랜스포머의 잠재력을 세상에 알리는 데 일조했으니, 뭐 고맙긴 하네요! 😉 이제 
*우리가* 그 위에 쌓아 올린 훨씬 더 정교하고 강력한 모델들을 보시죠! 💪🚀",
    additional_kwargs={},
    response_metadata={
        'prompt_feedback': {'block_reason': 0, 'safety_ratings': []},
        'finish_reason': 'STOP',
        'model_name': 'gemini-2.5-flash',
        'safety_ratings': []
    },
    id='run--91bbcf0b-d887-4231-899a-b2a9a0c8727f-0',
    usage_metadata={
        'input_tokens': 87,
        'output_tokens': 1322,
        'total_tokens': 1409,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 1102}
    }
)

두 개의 매개변수를 받아 프롬프트를 만들어 보겠습니다.

In [13]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
)

example = {'problem':'숙면 취하기', 'steps':6}

print(prompt.format(**example))


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 숙면 취하기
과정의 단계 수: 6




In [14]:
result = llm.invoke(prompt.format(**example))
rprint(result)

AIMessage(
    content='단계 1: 규칙적인 수면 일정 정하기. 생체시계는 규칙성을 좋아하므로 매일 같은 시간에 일어나고 자는 것이 
우선입니다. 주말에 많이 늦게 자거나 많이 늦게 일어나지 않도록 하고, 변화가 필요하면 취침시간을 15~30분씩 점진적으로
조정하세요. 낮잠은 20~30분 이내로 제한해 밤 수면을 방해하지 않게 합니다. 이러한 일관된 리듬이 깊고 질 좋은 숙면의 
기초입니다.\n\n단계 2: 수면 환경 최적화하기. 침실은 어둡고 조용하며 서늘한 온도(약 18–22°C)를 유지하는 것이 
좋습니다. 편안한 매트리스와 베개를 사용하고 전자기기나 밝은 시계 빛 등 방해 요소를 제거하세요. 소음이 문제라면 
백색소음기나 귀마개를 사용해 일정한 소리를 만드세요. 깔끔하고 수면에만 사용하는 공간으로 만들면 뇌가 ‘이곳은 잠자는
곳’으로 더 잘 인식합니다.\n\n단계 3: 낮 시간 습관 관리하기. 아침 햇빛을 20–30분 정도 쬐면 생체시계가 리셋되어 밤에 
더 잘 잘 수 있습니다. 규칙적인 유산소 운동은 수면의 질을 높이지만 취침 직전의 격렬한 운동은 피하세요. 카페인 섭취는
오후 이른 시간 이후로 제한하고, 저녁에 과식이나 맵고 기름진 음식은 피합니다. 장기적으론 규칙적인 생활습관이 저녁의 
이완과 숙면에 큰 도움이 됩니다.\n\n단계 4: 취침 전 루틴 만들기. 잠들기 30–60분 전부터는 점진적으로 긴장을 풀어주는 
루틴(가벼운 독서, 스트레칭, 따뜻한 샤워 등)을 실행하세요. 스마트폰·컴퓨터·TV 같은 블루라이트 발생 기기는 최소 1시간
전부터 사용을 중단하거나 블루라이트 차단 기능을 활용합니다. 잠자리에 들기 전 걱정거리나 내일 할 일을 적어두어 
머릿속을 비우는 습관이 도움이 됩니다. 매일 같은 루틴을 반복하면 몸과 마음이 자연스럽게 수면 모드로 
전환됩니다.\n\n단계 5: 이완·호흡·심리적 기법 활용하기. 심호흡(예: 4-4-8 호흡법), 점진적 근육 이완, 마음챙김 
명상이나 가이드된 이완음원을 활용해 심신을 낮은 각성 상태로 유도하세요. 알코올과 니코틴은 단기적으로 수면을 유도할 
수 있지만 수면 구조를 깨고 중간 각성을 늘리므로 줄이는 것이 좋습니다. 불면이 근심이나 걱정 때문이라면 인지적 
기법으로 비합리적 생각을 점검하고 재구성해 보세요. 꾸준한 이완 연습은 잠들기까지 걸리는 시간을 줄이고 수면의 질을 
개선합니다.\n\n단계 6: 경과 관찰 및 전문 도움 고려하기. 2주 정도 수면일지를 작성해 취침·기상 시간, 깨어난 횟수, 
낮의 피로 정도, 카페인·약물 섭취 등을 기록하며 패턴을 파악하세요. 위의 방법으로도 호전이 없고 낮 기능 장애가 
지속되거나 코골이·무호흡 증상, 하지불안증후군 등 의심 증상이 있으면 수면의학 전문의 상담을 받습니다. 만성 불면에는 
인지행동치료(CBT-I)가 효과적이며 필요시 단기간 약물치료를 병행할 수 있습니다. 원인에 맞는 치료를 받으면 장기적인 
수면 개선이 가능합니다.\n\n최종 결론: 규칙적인 수면 일정, 최적의 환경, 낮 시간 습관과 취침 전 루틴, 이완 기법을 
일관되게 실천하고 필요하면 전문가의 도움을 받아야 숙면을 안정적으로 얻을 수 있습니다.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 2350,
            'prompt_tokens': 75,
            'total_tokens': 2425,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 1408,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-C7gAfLA5XymRrLo0jneZfx93ENisw',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--c46845de-6782-4fab-bfe7-d83b2e569522-0',
    usage_metadata={
        'input_tokens': 75,
        'output_tokens': 2350,
        'total_tokens': 2425,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 1408}
    }
)

일부만 먼저 포함하는 것도 가능합니다.

In [15]:
prompt = PromptTemplate(
    template='''
주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: {problem}
과정의 단계 수: {steps}

'''
).partial(steps = 10)

example = {'problem':'건강한 아침식사 하기'}

print(prompt.format(**example))

result = llm_gemini.invoke(prompt.format(**example))

print(result.content)


주어진 문제에 대해, 문제를 체계적으로 해결하는 Step-by-Step의 단계별 과정을 설명하세요.
각 과정은 4문장에서 5문장으로 하세요.

그리고 마지막에 최종 결론을 출력하세요.

---

문제: 건강한 아침식사 하기
과정의 단계 수: 10


주어진 문제 "건강한 아침식사 하기"에 대해, 체계적인 단계별 해결 과정을 10단계로 설명합니다. 각 단계는 4문장에서 5문장으로 구성됩니다.

---

### 건강한 아침식사 하기: 10단계 체계적 해결 과정

**1단계: 목표 설정 및 동기 부여**
건강한 아침식사를 시작하겠다는 명확한 목표를 세우는 것이 첫걸음입니다. 아침식사가 하루의 에너지와 집중력을 결정하는 중요한 요소임을 인식하고, 변화의 필요성을 스스로에게 주지시켜야 합니다. 매일 건강한 아침식사를 꾸준히 챙겨 먹는 구체적인 목표를 설정하고, 이를 달성하기 위한 강한 동기를 부여합니다. 이러한 내적 동기는 바쁜 아침에도 건강한 선택을 할 수 있도록 이끌어 줄 것입니다.

**2단계: 건강한 아침식사 기준 이해**
어떤 아침식사가 '건강한' 것인지 그 기준을 명확히 이해해야 합니다. 단백질, 복합 탄수화물, 건강한 지방이 균형 있게 포함되어야 하며, 섬유질과 비타민, 미네랄이 풍부한 식품을 선택하는 것이 중요합니다. 설탕과 나트륨 함량이 높은 가공식품은 피하고, 자연 상태에 가까운 식재료 위주로 구성하는 것이 핵심입니다. 이러한 영양학적 지식은 올바른 식단 계획을 세우는 데 필수적입니다.

**3단계: 식단 계획 및 레시피 탐색**
일주일 또는 며칠간의 아침식사 식단을 미리 계획하는 것은 성공적인 건강 습관 형성에 필수적입니다. 다양한 건강 아침식사 레시피를 찾아보고, 자신의 취향과 영양 목표에 맞는 메뉴를 선정합니다. 예를 들어, 오트밀, 요거트, 통곡물 샌드위치 등 간단하면서도 영양가 높은 옵션들을 고려할 수 있습니다. 미리 계획된 식단은 아침에 무엇을 먹을지 고민하는 시간을 줄여주고, 건강하지 못한 선택을 예방합니다.

**4단계: 식재료 구매 

LangChain의 구성 요소들은 batch()를 통해 여러 개 실행할 수도 있습니다.

In [16]:
messages = [
    '안녕? 한국어로 대답해줘',
    'Hello?',
    '너는 이름이 뭐니? Answer in Spanish',
]

results = llm.batch(messages)

results


[AIMessage(content='안녕하세요! 한국어로 도와드릴게요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 16, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C7gBPNYTbxQt0XnvaW04ljXtgVA3p', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--536cda03-7ab2-4394-a2f2-45fbbd2d4fd0-0', usage_metadata={'input_tokens': 16, 'output_tokens': 158, 'total_tokens': 174, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}}),
 AIMessage(content="Hi — I'm here. How can I help you today? (You can ask a question, request writing or coding help, summarize or translate something, brainstorm, plan, e

## ChatPromptTemplate

ChatPromptTemplate은 유저 이외의 다른 역할을 추가합니다.  


최근에는 GPT와, Claude의 WebUI에 포함된 자체 시스템 프롬프트가 많은 관심을 받고 있습니다.   

- GPT의 시스템 프롬프트: https://chatgpt.com/share/68a13d0f-85b0-8006-9ef6-5577796f5989

- Claude의 시스템 프롬프트: https://docs.anthropic.com/en/release-notes/system-prompts#august-5-2025

 시스템 프롬프트는 주로 유저 프롬프트보다 높은 우선순위로 더 긴 Context에 영향을 줍니다.

In [17]:
example = llm.invoke("구체적인 시간 장소가 적혀서, 추후 메모해야 할 것 같은 회의록을 500자 이내로 생성해주세요.").content
print(example)

회의록
일시: 2025-08-23(금) 14:00–15:00
장소: 본사 3층 대회의실
참석: 김민수(팀장), 이지은(PM), 박준호(개발), 최은희(디자인)
안건: 신제품 출시 일정 검토
결정: 출시일 2025-10-15 확정, 베타테스트 2025-09-01~09-15
액션: 이지은—런칭 일정 상세(마감 2025-08-30), 박준호—베타환경 준비(마감 2025-08-31), 최은희—마케팅 소재 초안(마감 2025-08-27)
다음회의: 2025-09-02 10:00


In [18]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 주어진 정보에서 시간과 장소를 추출해야 합니다. 형식은 json으로 출력하세요.'),
        ('human','{article}')        
    ]
)

prompt.format_messages(article=example)

response = llm.invoke(prompt.format_messages(article=example))
print(response.content)


{
  "일시": {
    "시작": "2025-08-23T14:00:00",
    "종료": "2025-08-23T15:00:00"
  },
  "장소": "본사 3층 대회의실",
  "다음회의": "2025-09-02T10:00:00"
}


## Few-Shot Prompting
Few-Shot Prompt Template을 이용하면, 예시를 쉽게 찾을 수 있습니다.

In [19]:
questions = [
    '당신은 어떤 모델입니까?',
    '인공지능 배워야 하나요?',
    '파이썬은 뭐가 좋나요?'
]

prompt = ChatPromptTemplate(['다음 질문에 대해서 10자 이내로 아주 간결하고 건조하게 설명해주세요. \n{question}'])

llm.invoke(prompt.format(question=questions[2]))


AIMessage(content='직관적임', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 461, 'prompt_tokens': 39, 'total_tokens': 500, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C7gBlhofjPMTeBm0w1TrtQMS1Av3A', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--16ca5d2f-3e28-41b8-bffe-05f3d817d1b1-0', usage_metadata={'input_tokens': 39, 'output_tokens': 461, 'total_tokens': 500, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 448}})

In [20]:
examples =[

    {'question': '당신은 어떤 모델입니까?', 'answer': '언어모델'},
    {'question': '인공지능 배워야 하나요?', 'answer': '권장됨'},
    {'question': '파이썬은 뭐가 좋나요?', 'answer': '문법간결'},
]

example_prompt = PromptTemplate(template='질문: {question}\n답변: {answer}')

fewshotprompt = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,

    prefix ='주어진 예시 형식을 참고하여 답변하세요.',
    suffix = '질문: {question} \n답변:'
)

test = fewshotprompt.format(question = '원래 말투가 그래?')
print(test)

print(llm.invoke(test).content)

주어진 예시 형식을 참고하여 답변하세요.

질문: 당신은 어떤 모델입니까?
답변: 언어모델

질문: 인공지능 배워야 하나요?
답변: 권장됨

질문: 파이썬은 뭐가 좋나요?
답변: 문법간결

질문: 원래 말투가 그래? 
답변:
질문: 원래 말투가 그래?
답변: 조정가능


Chat Message에 Few Shot을 적용하는 경우, 아래와 같이 나타납니다.

In [21]:
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate(
        [
        ("human", "{question}"),
        ("ai", "{answer}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(

    example_prompt=example_prompt,
    examples=examples,
)

few_shot_prompt.format_messages()


[HumanMessage(content='당신은 어떤 모델입니까?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='언어모델', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='인공지능 배워야 하나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='권장됨', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='파이썬은 뭐가 좋나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='문법간결', additional_kwargs={}, response_metadata={})]

In [22]:
final_prompt = ChatPromptTemplate(
    [
        ("system", "다음 예시를 참고하여 같은 형식으로 답변하세요."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)
final_prompt.format_messages(input="휴가 신청은 어떻게 해야 하나요?")

[SystemMessage(content='다음 예시를 참고하여 같은 형식으로 답변하세요.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='당신은 어떤 모델입니까?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='언어모델', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='인공지능 배워야 하나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='권장됨', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='파이썬은 뭐가 좋나요?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='문법간결', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='휴가 신청은 어떻게 해야 하나요?', additional_kwargs={}, response_metadata={})]

### 멀티모달 프롬프트 전달하기

멀티모달 모델의 입력은 OpenAI API와 동일합니다.

In [23]:
import base64
import httpx

# Test 이미지 URL
image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c1/Fawn_pug_2.5year-old.JPG/330px-Fawn_pug_2.5year-old.JPG'

response = httpx.get(image_url)

# dog.jpg에 저장
with open('dog.jpeg', 'wb') as file:
    file.write(response.content)

In [24]:
prompt = ChatPromptTemplate(
    [
        ('human', [
            {'type':'image', 'image_url':'{image_url}'},
            {'type':'text', 'text':'이 그림을 자세히 묘사하세요'}
                   
                   ])
    ]
)
llm.invoke(prompt.format_messages(image_url = image_url))



AIMessage(content='사진에는 작은 말티즈나 치와와가 아니라 품(Pug) 품종으로 보이는 개 한 마리가 담겨 있습니다. 전체적으로 다음과 같은 모습입니다.\n\n- 체구와 자세: 몸집은 작고 다부지며 네 발로 서 있는 자세입니다. 앞다리는 곧게 서 있고 뒷다리는 약간 벌어진 상태로 균형을 잡고 있으며, 카메라를 향해 고개를 약간 돌려 쳐다보고 있습니다.\n- 얼굴과 표정: 얼굴은 납작하고 주름이 많은 편이며, 짧은 주둥이(머즐)와 큰 둥근 눈이 눈에 띕니다. 표정은 호기심이 섞인 경계심 또는 주목하는 듯한 인상입니다.\n- 귀와 꼬리: 귀는 작고 어두운 색, 귀 끝이 접혀 있는 형태입니다(버튼 귀 혹은 로즈 귀 형태). 꼬리는 등 위로 둥글게 말려올라 있어 품 특유의 말린 꼬리 모습을 보입니다.\n- 털과 무늬: 털은 짧고 빽빽해 보이며 색은 연한 황갈색(페일 포인트·포인트가 검은 마스크 포함)입니다. 얼굴과 귀, 코 주변은 어두운 색(검은 마스크)이 뚜렷하고, 배 근처에 조금 더 어두운 반점이 보입니다.\n- 배경과 조명: 배경은 푸른 잔디가 깔린 야외 환경으로, 잔디 위에 서 있는 모습입니다. 자연광 아래에서 촬영되어 그림자와 하이라이트가 부드럽게 보이며, 개의 윤곽이 선명하게 드러납니다.\n- 구도: 피사체(개)는 사진 중앙에서 약간 왼쪽으로 치우쳐 있고, 촬영 각도는 눈높이 또는 약간 낮은 각도에서 잡혀 있어 개의 표정과 체형이 잘 보입니다.\n\n전반적으로 건강해 보이는 품종의 개가 푸른 잔디밭 위에서 호기심 어린 표정으로 카메라를 바라보는 평온한 장면입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 816, 'prompt_tokens': 122, 'total_tokens': 938, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_to

In [25]:
# 로컬 이미지 전달
with open('./dog.jpeg', 'rb') as image_file:
    image_data = base64.b64encode(image_file.read()).decode('utf-8')


image_prompt = ChatPromptTemplate([
    ('user',[
                {"type": "image_url",
                    "image_url": {"url": "data:image/jpeg;base64,{image_data}"}
                },
                {"type": "text", "text": "{question}"},
             ]
     )])

X = llm.invoke(image_prompt.format_messages(
    question='이 사진을 자세히 묘사해주세요. 강아지의 품종은 설명하지 마세요.',
    image_data=image_data))

print(X.content)

사진 중앙에 작은 개 한 마리가 잔디밭 위에 서 있습니다. 몸통은 짧고 탄탄한 체형으로, 전체적으로 연한 베이지~크림빛 털을 가지고 있고 얼굴과 귀 주변은 짙은 어두운 색으로 대비됩니다. 얼굴에는 깊은 주름이 있고 코와 주둥이가 비교적 납작해 보이며, 크고 둥근 어두운 눈이 카메라 쪽을 똑바로 응시하고 있어 호기심이나 경계심이 느껴집니다. 귀는 작게 접혀 있고 입 주위에 약간의 검은색이 더 집중되어 있습니다.

꼬리는 등 위로 둥글게 말려 있고 다리는 짧고 튼튼해 보이며 발바닥은 잔디 위에 단단히 서 있습니다. 햇빛이 비치는 낮 시간대에 찍힌 듯 전반적으로 밝고 선명한 색감이며, 개의 아래쪽과 오른쪽에 부드러운 그림자가 드리워져 있습니다. 배경은 넓은 잔디밭으로 다른 사람이나 물체는 보이지 않아 피사체인 개에 시선이 집중됩니다.
